# <div align="center"><u>The CDG-GS dataset</u>

## - General description of the CDG-GS dataset:
    
The following implementation applies an CDG-GS implementation to the Colon-Kidney dataset, where the dimensionality ranges between 55 and 110 features (gene expressions).

##############################################################################################################################

###  Import some useful libraries

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import random
from sklearn import metrics

from sklearn.metrics.pairwise import pairwise_distances
from scipy.sparse import csr_matrix

import networkx as nx

import time

### Suppress warnings

In [29]:
# import warnings

# # To suppress all warnings
# warnings.filterwarnings("ignore")

### Start time

In [30]:
# Start time
start_time = time.time()

### Load the final colon-kidney dataset and store it within a pandas DataFrame

In [31]:
# Load the dataset and store it withina  panda dataframe to handle it easier
df = pd.read_csv(r"C:\Users\user\Desktop\AUTH\Διπλωματική\Python\The colon-kidney final dataset.csv")

# Print the dataset
df

,1007_s_at,121_at,1405_i_at,1438_at,1487_at,1494_f_at,1552256_a_at,1552257_a_at,1552274_at,1552275_s_at,...,AFFX-r2-Ec-bioC-5_at,AFFX-r2-Ec-bioD-3_at,AFFX-r2-Ec-bioD-5_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,AFFX-ThrX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at,Target,Tissue
0,2883.2,2109.1,702.0,114.4,822.2,235.1,3346.1,815.1,233.3,218.4,...,1908.1,30396.7,28371.7,85830.1,57084.9,3651.2,1736.1,2411.6,0,Kidney
1,2607.4,1204.2,228.0,19.2,1497.1,167.6,23003.6,948.6,414.1,124.1,...,3012.0,16748.8,13581.2,53322.5,37139.5,1648.1,502.9,890.6,0,Kidney
2,2736.8,3912.2,1020.8,39.0,574.9,448.9,1337.9,605.6,176.8,411.9,...,14835.5,64597.8,59311.2,203060.6,146422.6,3346.4,673.2,1499.3,0,Kidney
3,1471.6,1318.2,4564.6,9.4,1426.5,105.2,2594.3,929.8,240.4,139.0,...,11300.8,58551.7,49073.6,119460.8,95682.3,2553.5,961.2,1468.4,0,Kidney
4,3564.7,2535.6,944.1,40.0,734.2,651.0,1490.5,712.3,632.7,185.0,...,12966.9,54572.5,51049.1,178178.2,129533.2,3399.1,1209.8,1981.6,0,Kidney
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,3197.8,713.4,191.1,747.0,1597.8,179.5,1717.2,2261.2,305.5,181.8,...,1494.9,22165.4,20442.0,65246.8,46710.2,2057.3,810.0,1338.2,1,Colon
542,2161.9,1244.3,2347.0,1995.6,697.8,194.6,508.4,1042.5,729.7,537.2,...,4588.2,21225.6,20067.2,80607.7,60204.9,1116.3,432.8,744.5,1,Colon
543,4105.2,664.4,445.2,7008.2,1887.0,204.0,1951.8,2540.1,259.2,239.5,...,1318.0,19659.4,15774.8,56065.6,43347.8,1191.2,371.3,688.5,1,Colon
544,4204.7,894.2,102.8,1374.8,2891.5,306.2,1397.9,2072.4,474.0,316.3,...,6375.2,34393.5,29137.7,89215.2,71290.1,1878.7,1009.7,1126.6,1,Colon


### Exclude last 2 columns (Targets)

In [32]:
# Exclude the last two columns
df_excluded = df.iloc[:, :-2]

df_excluded

,1007_s_at,121_at,1405_i_at,1438_at,1487_at,1494_f_at,1552256_a_at,1552257_a_at,1552274_at,1552275_s_at,...,AFFX-r2-Ec-bioB-M_at,AFFX-r2-Ec-bioC-3_at,AFFX-r2-Ec-bioC-5_at,AFFX-r2-Ec-bioD-3_at,AFFX-r2-Ec-bioD-5_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,AFFX-ThrX-3_at,AFFX-ThrX-5_at,AFFX-ThrX-M_at
0,2883.2,2109.1,702.0,114.4,822.2,235.1,3346.1,815.1,233.3,218.4,...,2386.3,2035.7,1908.1,30396.7,28371.7,85830.1,57084.9,3651.2,1736.1,2411.6
1,2607.4,1204.2,228.0,19.2,1497.1,167.6,23003.6,948.6,414.1,124.1,...,1225.7,3811.6,3012.0,16748.8,13581.2,53322.5,37139.5,1648.1,502.9,890.6
2,2736.8,3912.2,1020.8,39.0,574.9,448.9,1337.9,605.6,176.8,411.9,...,5114.2,17154.4,14835.5,64597.8,59311.2,203060.6,146422.6,3346.4,673.2,1499.3
3,1471.6,1318.2,4564.6,9.4,1426.5,105.2,2594.3,929.8,240.4,139.0,...,4579.8,12382.2,11300.8,58551.7,49073.6,119460.8,95682.3,2553.5,961.2,1468.4
4,3564.7,2535.6,944.1,40.0,734.2,651.0,1490.5,712.3,632.7,185.0,...,5666.9,13321.1,12966.9,54572.5,51049.1,178178.2,129533.2,3399.1,1209.8,1981.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,3197.8,713.4,191.1,747.0,1597.8,179.5,1717.2,2261.2,305.5,181.8,...,1425.0,1752.1,1494.9,22165.4,20442.0,65246.8,46710.2,2057.3,810.0,1338.2
542,2161.9,1244.3,2347.0,1995.6,697.8,194.6,508.4,1042.5,729.7,537.2,...,1663.2,5334.5,4588.2,21225.6,20067.2,80607.7,60204.9,1116.3,432.8,744.5
543,4105.2,664.4,445.2,7008.2,1887.0,204.0,1951.8,2540.1,259.2,239.5,...,1145.6,1511.6,1318.0,19659.4,15774.8,56065.6,43347.8,1191.2,371.3,688.5
544,4204.7,894.2,102.8,1374.8,2891.5,306.2,1397.9,2072.4,474.0,316.3,...,2280.7,7192.6,6375.2,34393.5,29137.7,89215.2,71290.1,1878.7,1009.7,1126.6


### Store the dataset in an array

In [33]:
X = df_excluded.values  # Convert DataFrame to NumPy array
M, N = X.shape # shape (M, N)

### Set number of neighbors and d (number of genes to select)

In [34]:
k = 3         # number of nearest neighbors
d = 100       # number of genes to select

# Initialize combined Laplacian
L_combined = np.zeros((M, M))

### Step 1: Loop over each gene to build per-gene Laplacian and accumulate

In [35]:
# Step 1: Loop over each gene to build per-gene Laplacian and accumulate
for i in range(N):
    gene_signal = X[:, i]  # shape (M,)

    # Compute pairwise squared Euclidean distances
    diff = gene_signal[:, np.newaxis] - gene_signal[np.newaxis, :]
    D = diff ** 2  # shape (M, M)

    # Build k-NN adjacency matrix
    A = np.zeros((M, M))
    for m in range(M):
        neighbors = np.argsort(D[m])[:k + 1]  # include self
        neighbors = neighbors[neighbors != m][:k]  # remove self
        A[m, neighbors] = 1

    # Symmetrize adjacency matrix
    A = np.maximum(A, A.T)

    # Compute degree and Laplacian
    Dg = np.diag(np.sum(A, axis=1))
    L_i = Dg - A

    # Accumulate Laplacian
    L_combined += L_i

### Step 2: Average the Laplacian

In [36]:
# Step 2: Average the Laplacian
L_combined /= N

### Step 3: Sparsify the Laplacian (keep only top-k values per column)

In [37]:
# Step 3: Sparsify the Laplacian (keep only top-k values per column)
L_sparse = np.zeros_like(L_combined)
for j in range(M):
    top_k_indices = np.argsort(-L_combined[:, j])[:k]
    L_sparse[top_k_indices, j] = L_combined[top_k_indices, j]

### Step 4: Make the Laplacian symmetric

In [38]:
# Step 4: Make it symmetric again (important)
L_sparse = np.maximum(L_sparse, L_sparse.T)

### Step 5: Compute roughness score for each gene

In [39]:
# Step 5: Compute roughness score for each gene
results = np.zeros(N)
for l in range(N):
    x_l = X[:, l]
    results[l] = x_l.T @ L_sparse @ x_l

### Step 6: Sort genes by descending roughness

In [40]:
# Step 6: Sort genes by descending roughness
sorted_indices = np.argsort(-results)
selected_indices = sorted_indices[:d]

### Step 7: Select columns from X

In [41]:
# Step 7: Select columns from X
Y = X[:, selected_indices]

### Store only the selected columns in the dataframe

In [42]:
# Get all column indices
all_indices = list(range(df.shape[1]))

# Get indices to drop = all except selected
drop_indices = [i for i in all_indices if i not in selected_indices and i < df.shape[1] - 2]

# Drop those columns
selected_df = df.drop(df.columns[drop_indices], axis=1)

df = selected_df

df

,1553538_s_at,1553551_s_at,1553567_s_at,1553569_at,1553570_x_at,1553588_at,1555653_at,200003_s_at,200018_at,200026_at,...,229563_s_at,AFFX-CreX-3_at,AFFX-CreX-5_at,AFFX-HSAC07/X00351_3_at,AFFX-hum_alu_at,AFFX-HUMGAPDH/M33197_3_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,Target,Tissue
0,101939.7,79188.2,78548.9,79695.4,81373.3,82932.5,48695.9,46503.7,44059.7,52231.1,...,53096.8,64127.8,59227.1,42515.8,90722.5,49946.3,85830.1,57084.9,0,Kidney
1,62075.5,53001.9,43285.7,50123.0,51560.9,49486.0,25368.1,29198.2,44469.3,30708.5,...,37691.6,33593.3,28815.0,49046.8,54776.9,54811.3,53322.5,37139.5,0,Kidney
2,168655.1,32053.3,156871.0,104305.2,133398.5,154276.8,72650.6,46739.0,38308.0,68655.5,...,70948.3,127790.6,108451.5,49118.3,214034.7,62308.8,203060.6,146422.6,0,Kidney
3,127382.8,106139.6,93669.1,107430.6,108189.8,141085.6,75596.0,53118.7,44688.8,49421.6,...,50917.5,98447.0,90216.3,65367.0,112892.8,69876.5,119460.8,95682.3,0,Kidney
4,202406.5,89194.2,134037.2,96200.1,111069.8,137204.3,83284.8,45815.7,47404.7,50099.5,...,58288.3,131028.8,118324.1,53624.2,191001.4,80397.2,178178.2,129533.2,0,Kidney
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,82650.3,70763.6,67911.5,81602.7,78553.4,79633.4,29272.7,60428.7,55766.2,41372.6,...,56906.0,46628.6,42320.0,54417.9,74103.6,45601.3,65246.8,46710.2,1,Colon
542,107124.6,76868.8,83311.6,65950.8,73512.4,85249.7,20338.2,54951.0,52572.2,56874.8,...,67786.0,60248.2,46485.4,64116.4,106527.6,51837.2,80607.7,60204.9,1,Colon
543,80942.7,62972.8,50785.5,73531.2,74250.5,74794.9,33890.7,52274.5,49449.9,47478.7,...,55027.6,38609.1,32498.5,46663.5,67663.2,40467.4,56065.6,43347.8,1,Colon
544,118584.1,103234.3,78075.0,112106.0,108282.6,105504.0,52147.5,58638.3,64498.7,65883.7,...,72677.2,65646.3,57327.1,54553.5,105755.9,57172.0,89215.2,71290.1,1,Colon


### Export the CDG-GS dataset to .csv file

In [43]:
# Export to .csv file
file_path = r"C:\Users\user\Desktop\AUTH\Διπλωματική\Python\Dimensionality Reduction Methods\Graph Based Methods\CDG-GS\The colon-kidney CDG-GS filtered dataset.csv"
df.to_csv(file_path, index=False)

print(f"CSV file saved at: {file_path}")

CSV file saved at: C:\Users\user\Desktop\AUTH\Διπλωματική\Python\Dimensionality Reduction Methods\Graph Based Methods\CDG-GS\The colon-kidney CDG-GS filtered dataset.csv


### Transform outliers

In [44]:
# Function that tranforms outliers given a dataframe
def cap_outliers_iqr(df, threshold=1.5):
    df_capped = df.copy()
    for col in df.columns:
        if col in df_capped.columns and pd.api.types.is_numeric_dtype(df_capped[col]):
            Q1 = df_capped[col].quantile(0.25)
            Q3 = df_capped[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - threshold * IQR
            upper_bound = Q3 + threshold * IQR

            df_capped[col] = np.where(df_capped[col] < lower_bound, lower_bound, df_capped[col])
            df_capped[col] = np.where(df_capped[col] > upper_bound, upper_bound, df_capped[col])
        else:
            print(f"Warning: Column '{col}' not found or is not numeric. Skipping.")
    return df_capped

In [45]:
# Transform outliers
df_capped = cap_outliers_iqr(df)
df_capped

,1553538_s_at,1553551_s_at,1553567_s_at,1553569_at,1553570_x_at,1553588_at,1555653_at,200003_s_at,200018_at,200026_at,...,229563_s_at,AFFX-CreX-3_at,AFFX-CreX-5_at,AFFX-HSAC07/X00351_3_at,AFFX-hum_alu_at,AFFX-HUMGAPDH/M33197_3_at,AFFX-r2-P1-cre-3_at,AFFX-r2-P1-cre-5_at,Target,Tissue
0,101939.7,79188.2,78548.9,79695.4,81373.3,82932.5,48695.9,46503.7,44059.7,52231.1,...,53096.8,64127.8,59227.1,42515.8,90722.5,49946.3,85830.1,57084.9,0.0,Kidney
1,62075.5,53001.9,43285.7,50123.0,51560.9,49486.0,25368.1,29198.2,44469.3,30708.5,...,37691.6,33593.3,28815.0,49046.8,54776.9,54811.3,53322.5,37139.5,0.0,Kidney
2,168655.1,32053.3,156871.0,104305.2,133398.5,154276.8,72650.6,46739.0,38308.0,68655.5,...,70948.3,127790.6,108451.5,49118.3,214034.7,62308.8,203060.6,146422.6,0.0,Kidney
3,127382.8,106139.6,93669.1,107430.6,108189.8,141085.6,75596.0,53118.7,44688.8,49421.6,...,50917.5,98447.0,90216.3,65367.0,112892.8,69876.5,119460.8,95682.3,0.0,Kidney
4,202406.5,89194.2,134037.2,96200.1,111069.8,137204.3,83284.8,45815.7,47404.7,50099.5,...,58288.3,131028.8,118324.1,53624.2,191001.4,80397.2,178178.2,129533.2,0.0,Kidney
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
541,82650.3,70763.6,67911.5,81602.7,78553.4,79633.4,29272.7,60428.7,55766.2,41372.6,...,56906.0,46628.6,42320.0,54417.9,74103.6,45601.3,65246.8,46710.2,1.0,Colon
542,107124.6,76868.8,83311.6,65950.8,73512.4,85249.7,20338.2,54951.0,52572.2,56874.8,...,67786.0,60248.2,46485.4,64116.4,106527.6,51837.2,80607.7,60204.9,1.0,Colon
543,80942.7,62972.8,50785.5,73531.2,74250.5,74794.9,33890.7,52274.5,49449.9,47478.7,...,55027.6,38609.1,32498.5,46663.5,67663.2,40467.4,56065.6,43347.8,1.0,Colon
544,118584.1,103234.3,78075.0,112106.0,108282.6,105504.0,52147.5,58638.3,64498.7,65883.7,...,72677.2,65646.3,57327.1,54553.5,105755.9,57172.0,89215.2,71290.1,1.0,Colon


### Check if "Transform Outliers" worked: Count how many elements are different between dataframes. If 0 nothing happened

In [46]:
# Compare element-wise and count differences
num_differences = (df != df_capped).sum().sum()

# Print results
if num_differences > 0:
    print(f"The tranformation of outliers worked. The amount of outliers that has been transformed is: {num_differences}")
else:
    print("No values transformed during outliers transformation phase")

The tranformation of outliers worked. The amount of outliers that has been transformed is: 1330


### Export the dataset to .csv file

In [47]:
# Export to .csv file
file_path = r"C:\Users\user\Desktop\AUTH\Διπλωματική\Python\Dimensionality Reduction Methods\Graph Based Methods\CDG-GS\The colon-kidney CDG-GS filtered capped dataset.csv"
df_capped.to_csv(file_path, index=False)

print(f"CSV file saved at: {file_path}")

CSV file saved at: C:\Users\user\Desktop\AUTH\Διπλωματική\Python\Dimensionality Reduction Methods\Graph Based Methods\CDG-GS\The colon-kidney CDG-GS filtered capped dataset.csv


### End time

In [48]:
# End time
end_time = time.time()

### Elapsed time

In [49]:
# Elapsed time
elapsed_time = end_time - start_time

print(f"Elapsed time for CDG-GS Dataset: {elapsed_time:.2f} seconds")

Elapsed time for CDG-GS Dataset: 260.28 seconds
